# Jacobian scratch pad

Manual experiments on the t=2 FutureINN step. `forward_step` advances `(high, mid, low)`; `ad_jvp` returns the packed next state and the exact Jacobian-vector product via `torch.func.jvp`.

**Run on a GPU node.** Calling `ema_vit` is too heavy for a login node.

In [17]:
import argparse
import glob
import os
import sys
from pathlib import Path

_here = Path.cwd().resolve()
_repo = next(p for p in [_here, *_here.parents] if (p / "model.py").is_file())
if str(_repo) not in sys.path:
    sys.path.insert(0, str(_repo))

import numpy as np
import scipy.io
import torch
from torch import nn
from torch.nn import functional as F
import scipy
import scipy.io
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
from model import FutureINN
# parser = argparse.ArgumentParser(description="Jacobian scratch pad")
# parser.add_argument("--total-fields", type=int, default=10)
# parser.add_argument(
#     "--every-nth-field",
#     type=int,
#     default=5,
#     help="Keep a frame when its integer stem is divisible by this",
# )
# parser.add_argument("--eval-dir", default="/global/homes/d/divyam/TACSLAB/eval")
# parser.add_argument(
#     "--checkpoint",
#     default="/pscratch/sd/d/divyam/hine_runs/paper_1gpu/checkpoint-last.pth",
# )
# parser.add_argument(
#     "--model-type",
#     choices=["hine"],
#     default="hine",
#     help="Checkpoint class. Only the hierarchical implicit neural emulator for now.",
# )
# parser.add_argument(
#     "--n-initial-conditions",
#     type=int,
#     default=None,
#     help="How many initial conditions to test. Omit to use all that fit --horizon.",
# )
# parser.add_argument(
#     "--horizon",
#     type=int,
#     default=20,
#     help="Lead times to forecast from each initial condition",
# )
# parser.add_argument(
#     "--norm",
#     type=float,
#     default=10,
#     help="Normalization constant for the fields",
# )
# parser.add_argument(
#     "--device",
#     choices=["cpu", "cuda"],
#     default="cpu",
#     help="Device to run the model on",
# )
# parser.add_argument(
#     "--save-dir",
#     default="/pscratch/sd/d/divyam/hine_runs/paper_1gpu/jacobian_manual_mse",
#     help="Directory for the npz, the field figures, and the MSE curves",
# )
# args, _unknown = parser.parse_known_args()

# total_fields = args.total_fields
# every_nth_field = args.every_nth_field
# eval_dir = args.eval_dir
# checkpoint_path = args.checkpoint
# model_type = args.model_type
# n_initial_conditions = args.n_initial_conditions
# horizon = args.horizon
# norm = args.norm
# device_type = torch.device(args.device)
# save_dir = args.save_dir


total_fields = 10
every_nth_field = 5
eval_dir = "/global/homes/d/divyam/TACSLAB/eval"
checkpoint_path = "/pscratch/sd/d/divyam/hine_runs/paper_1gpu/checkpoint-last.pth"
model_type = "hine"
n_initial_conditions = 1
horizon = 20
norm = 10
device_type = "cpu"
device = torch.device(device_type)
save_dir = "/pscratch/sd/d/divyam/hine_runs/paper_1gpu/jacobian_manual_mse"


print(model_type)
if model_type != "hine":
    raise NotImplementedError(f"model type {model_type} is not wired up yet")


numbered = []
for path in glob.glob(os.path.join(eval_dir, "*.mat")):
    try:
        numbered.append((int(Path(path).stem), path))
    except ValueError:
        continue
numbered.sort(key=lambda item: item[0])
kept = numbered[::every_nth_field]
n_available = len(kept) // horizon
if n_available < 1:
    raise ValueError(
        f"Need at least {horizon} frames after taking every {every_nth_field}th, "
        f"found {len(kept)} from {len(numbered)} sorted files."
    )
if n_initial_conditions is None:
    chosen = np.arange(n_available)
else:
    if n_initial_conditions > n_available:
        raise ValueError(
            f"Asked for {n_initial_conditions} sections, only {n_available} fit."
        )
    chosen = (np.arange(n_initial_conditions) * n_available) // n_initial_conditions

trajectories = []
for block in chosen:
    start = int(block) * horizon
    frames = [
        scipy.io.loadmat(kept[start + k][1])["Omega"]
        for k in range(horizon)
    ]
    trajectories.append(frames)

print(
    f"sorted={len(numbered)} every_{every_nth_field}th={len(kept)} "
    f"full_sections={n_available} using={len(chosen)} "
    f"section_index={chosen.tolist()}"
)

hine
sorted=13380 every_5th=2676 full_sections=133 using=1 section_index=[0]


In [18]:

def ad_jvp(ema_vit, high, mid, low, v):
    """Exact JVP of the packed t=2 map via torch.func.jvp."""
    shapes = (high.shape, mid.shape, low.shape)
    n_h = high.numel()
    n_m = mid.numel()
    v_high = v[:n_h].reshape(shapes[0])
    v_mid = v[n_h : n_h + n_m].reshape(shapes[1])
    v_low = v[n_h + n_m :].reshape(shapes[2])

    def step(h, m, l):
        t = 2 * torch.ones(h.shape[0], device=h.device, dtype=h.dtype)
        return ema_vit(h, m, l, t)

    (y_h, y_m, y_l), (jy_h, jy_m, jy_l) = torch.func.jvp(
        step, (high, mid, low), (v_high, v_mid, v_low)
    )
    y = torch.cat([y_h.reshape(-1), y_m.reshape(-1), y_l.reshape(-1)])
    jv = torch.cat([jy_h.reshape(-1), jy_m.reshape(-1), jy_l.reshape(-1)])
    return y, jv

LOADING MODEL WEIGHTS

In [19]:
ckpt = torch.load(checkpoint_path, map_location=device_type, weights_only=False)

model = FutureINN(ckpt["args"])
model.load_state_dict(ckpt["model"])
model.eval()
ema_vit = model.ema_vit  # this is what forward_step / ad_jvp take

model = model.to(device)

LOADING DATA

In [20]:
fields = trajectories[0]

init_field = torch.from_numpy(np.asarray(fields[0])).to(device)
init_field = init_field.unsqueeze(0).unsqueeze(0)
init_field.shape

torch.Size([1, 1, 256, 256])

Rollout MSE

Batched autoregressive and teacher-forced rollouts. Each step records high, mid, and low mean-square error. Results and figures go to `save_dir`.

In [21]:


def downsample(x, factor):
    """Downsample by factor by average pooling."""
    return F.avg_pool2d(x, kernel_size=factor, stride=factor)
    

In [ ]:
os.makedirs(save_dir, exist_ok=True)
os.makedirs(os.path.join(save_dir, "fields"), exist_ok=True)

batch_size = len(trajectories)
n_frames = len(trajectories[0])
n_steps = n_frames - 3
if n_steps < 1:
    raise ValueError(f"Need at least 4 frames per section, found {n_frames}")

frames_np = np.stack(
    [np.stack(traj, axis=0) for traj in trajectories], axis=0
).astype(np.float32) / np.float32(norm)

file_stem = np.zeros((len(trajectories), n_frames), dtype=np.int64)
kept_index = np.zeros_like(file_stem)
for b, block in enumerate(chosen):
    start = int(block) * horizon
    for k in range(n_frames):
        kept_index[b, k] = start + k
        file_stem[b, k] = kept[start + k][0]
lead = np.arange(1, n_steps + 1, dtype=np.int64)
target_stem = file_stem[:, 1:n_steps + 1]
target_kept_index = kept_index[:, 1:n_steps + 1]

def spatial_mse(pred, target):
    return (pred - target).pow(2).mean(dim=(1, 2, 3)).double().cpu().numpy()

@torch.no_grad()
def rollout(teacher_forced):
    pred_high, pred_mid, pred_low = [], [], []
    gt_high, gt_mid, gt_low = [], [], []
    mse_high, mse_mid, mse_low = [], [], []
    ema_vit.eval()
    for start in range(0, frames_np.shape[0], batch_size):
        batch = torch.from_numpy(frames_np[start:start + batch_size]).to(device)
        batch = batch.unsqueeze(2)
        b = batch.shape[0]
        high = batch[:, 0]
        mid0 = torch.zeros(b, 1, model.ds_res, model.ds_res, device=device, dtype=high.dtype)
        low0 = torch.zeros(b, 1, model.low_res, model.low_res, device=device, dtype=high.dtype)
        _, mid, _ = ema_vit(high, mid0, low0, torch.zeros(b, device=device, dtype=high.dtype))
        _, _, low = ema_vit(high, mid, low0, torch.ones(b, device=device, dtype=high.dtype))
        t2 = 2 * torch.ones(b, device=device, dtype=high.dtype)
        hs, ms, ls, gh, gm, gl = [], [], [], [], [], []
        eh = np.zeros((b, n_steps), dtype=np.float64)
        em = np.zeros((b, n_steps), dtype=np.float64)
        el = np.zeros((b, n_steps), dtype=np.float64)
        for k in range(n_steps):
            if teacher_forced:
                high = batch[:, k]
            high, mid, low = ema_vit(high, mid, low, t2)
            true_h = batch[:, k + 1]
            true_m = F.avg_pool2d(batch[:, k + 2], kernel_size=model.ds_factor, stride=model.ds_factor)
            true_l = F.avg_pool2d(batch[:, k + 3], kernel_size=model.ds_factor_low, stride=model.ds_factor_low)
            eh[:, k] = spatial_mse(high, true_h)
            em[:, k] = spatial_mse(mid, true_m)
            el[:, k] = spatial_mse(low, true_l)
            hs.append(high.detach().float().cpu())
            ms.append(mid.detach().float().cpu())
            ls.append(low.detach().float().cpu())
            gh.append(true_h.detach().float().cpu())
            gm.append(true_m.detach().float().cpu())
            gl.append(true_l.detach().float().cpu())
        pred_high.append(torch.stack(hs, 1).numpy())
        pred_mid.append(torch.stack(ms, 1).numpy())
        pred_low.append(torch.stack(ls, 1).numpy())
        gt_high.append(torch.stack(gh, 1).numpy())
        gt_mid.append(torch.stack(gm, 1).numpy())
        gt_low.append(torch.stack(gl, 1).numpy())
        mse_high.append(eh)
        mse_mid.append(em)
        mse_low.append(el)
    cat = lambda xs: np.concatenate(xs, 0)
    return tuple(cat(xs) for xs in (
        pred_high, pred_mid, pred_low, gt_high, gt_mid, gt_low, mse_high, mse_mid, mse_low
    ))

(
    ar_pred_high, ar_pred_mid, ar_pred_low,
    gt_high, gt_mid, gt_low,
    ar_mse_high, ar_mse_mid, ar_mse_low,
) = rollout(False)
(
    tf_pred_high, tf_pred_mid, tf_pred_low,
    _, _, _,
    tf_mse_high, tf_mse_mid, tf_mse_low,
) = rollout(True)

npz_path = os.path.join(save_dir, "rollout_mse.npz")
np.savez_compressed(
    npz_path,
    ar_pred_high=ar_pred_high, ar_pred_mid=ar_pred_mid, ar_pred_low=ar_pred_low,
    tf_pred_high=tf_pred_high, tf_pred_mid=tf_pred_mid, tf_pred_low=tf_pred_low,
    gt_high=gt_high, gt_mid=gt_mid, gt_low=gt_low,
    ar_mse_high=ar_mse_high, ar_mse_mid=ar_mse_mid, ar_mse_low=ar_mse_low,
    tf_mse_high=tf_mse_high, tf_mse_mid=tf_mse_mid, tf_mse_low=tf_mse_low,
    lead=lead, file_stem=file_stem, target_stem=target_stem,
    kept_index=kept_index, target_kept_index=target_kept_index,
    chosen=np.asarray(chosen),
    checkpoint=np.asarray(checkpoint_path), eval_dir=np.asarray(eval_dir),
)
print(
    f"saved {npz_path}  B={ar_mse_high.shape[0]} T={ar_mse_high.shape[1]}  "
    f"lead1 high/mid/low AR "
    f"{ar_mse_high[:, 0].mean():.4g} / {ar_mse_mid[:, 0].mean():.4g} / {ar_mse_low[:, 0].mean():.4g}"
)

In [ ]:
plot_leads = [0, n_steps // 2, n_steps - 1]

def field_figure(b, k):
    rows = [
        ("high", ar_pred_high[b, k, 0], tf_pred_high[b, k, 0], gt_high[b, k, 0]),
        ("mid", ar_pred_mid[b, k, 0], tf_pred_mid[b, k, 0], gt_mid[b, k, 0]),
        ("low", ar_pred_low[b, k, 0], tf_pred_low[b, k, 0], gt_low[b, k, 0]),
    ]
    fig, axs = plt.subplots(3, 5, figsize=(16, 8), constrained_layout=True)
    titles = ["autoregressive", "teacher forced", "ground truth", "|AR − GT|", "|TF − GT|"]
    for r, (name, ar, tf, gt) in enumerate(rows):
        field_lim = (min(ar.min(), tf.min(), gt.min()), max(ar.max(), tf.max(), gt.max()))
        panels = [ar, tf, gt, np.abs(ar - gt), np.abs(tf - gt)]
        for c, panel in enumerate(panels):
            kw = {"vmin": field_lim[0], "vmax": field_lim[1]} if c < 3 else {}
            im = axs[r, c].imshow(panel, **kw)
            axs[r, c].set_xticks([])
            axs[r, c].set_yticks([])
            fig.colorbar(im, ax=axs[r, c], fraction=0.046)
            if r == 0:
                axs[r, c].set_title(titles[c])
            if c == 0:
                axs[r, c].set_ylabel(name)
    fig.suptitle(
        f"section {int(chosen[b])}  lead {int(lead[k])}  stem {int(target_stem[b, k])}"
    )
    return fig

for b in range(ar_mse_high.shape[0]):
    for k in plot_leads:
        fig = field_figure(b, k)
        fig.savefig(os.path.join(save_dir, "fields", f"b{b:03d}_lead{int(lead[k]):03d}.png"), dpi=120)
        if b == 0:
            plt.show()
        plt.close(fig)

In [ ]:
series = [
    ("high", ar_mse_high, tf_mse_high),
    ("mid", ar_mse_mid, tf_mse_mid),
    ("low", ar_mse_low, tf_mse_low),
]
fig, axs = plt.subplots(1, 3, figsize=(14, 4), sharex=True, constrained_layout=True)
for ax, (name, ar, tf) in zip(axs, series):
    for row in range(ar.shape[0]):
        ax.plot(lead, ar[row], color="C0", alpha=0.3)
        ax.plot(lead, tf[row], color="C1", alpha=0.3)
    ax.plot(lead, ar.mean(0), color="C0", lw=2.5, label="autoregressive")
    ax.plot(lead, tf.mean(0), color="C1", lw=2.5, label="teacher forced")
    ax.set_title(name)
    ax.set_xlabel("lead")
    ax.set_ylabel("MSE")
    ax.legend()
fig.savefig(os.path.join(save_dir, "mse_vs_lead.png"), dpi=150)
plt.show()


COMPUTING THE JACOBIAN FULL

In [ ]:

dtype = next(model.ema_vit.parameters()).dtype
initial_field = (init_field / norm).detach().to(dtype)
h0 = initial_field.detach()
mid, low = None, None
_, mid, low = model.sample(initial_field, mid, low, ema=True)
mid_c = mid.detach().to(dtype)
low_c = low.detach().to(dtype)
pred = model.sample(initial_field, mid, low, ema=True)[0]

model.eval()
t = 2 * torch.ones(initial_field.shape[0], device=initial_field.device)
J = torch.autograd.functional.jacobian(
    lambda h: ema_vit(h, mid, low, t)[0][0, 0, :, :],
    initial_field,
)[:, :, 0, 0, :, :]
pred, J

computing just the eigenvalues of the jacobian

In [ ]:
# model.eval()
# dtype = next(model.ema_vit.parameters()).dtype
# h0 = initial_field.detach().to(dtype)
# mid_c = mid.detach().to(dtype)
# low_c = low.detach().to(dtype)
# t = 2 * torch.ones(h0.shape[0], device=h0.device, dtype=dtype)
# v = torch.randn_like(h0)
# v = v / v.norm()

# for i in range(15):
#     _, jv = torch.autograd.functional.jvp(
#         lambda h: model.ema_vit(h, mid_c, low_c, t)[0],
#         (h0,),
#         (v,),
#     )
#     radius = jv.norm()
#     v = (jv / radius).detach()
#     print(i, float(radius))

In [ ]:
# import numpy as np
# from scipy.sparse.linalg import LinearOperator, eigs

# model.eval()
# dtype = next(model.ema_vit.parameters()).dtype
# h0 = initial_field.detach().to(dtype)
# mid_c = mid.detach().to(dtype)
# low_c = low.detach().to(dtype)
# t = 2 * torch.ones(h0.shape[0], device=h0.device, dtype=dtype)
# v = torch.randn_like(h0)
# v = v / v.norm()
# def matvec(v_np):
#     v = torch.from_numpy(np.ascontiguousarray(v_np)).to(device=h0.device, dtype=h0.dtype).reshape_as(h0)
#     _, jv = torch.autograd.functional.jvp(
#         lambda h: model.ema_vit(h, mid_c, low_c, t)[0],
#         (h0,),
#         (v,),
#     )
#     return jv.detach().reshape(-1).float().cpu().numpy()

# vals = eigs(LinearOperator((h0.numel(), h0.numel()), matvec=matvec, dtype=np.float32), k=6, which="LM", return_eigenvectors=False)
# radius = float(np.abs(vals).max())
# vals, radius